## Open Prices summer 2025 challenge completed !

The summer has ended and so did the 3rd Open Prices community challenge ! 

As always, we were looking for contributors to collect prices on a specific type of products. With the shining sun and blistering heat in parts of the world, the topics were: sunscreens, ice creams & sorbets! [Check out the challenge page](https://prices.openfoodfacts.org/challenges/3) for more info.

![](https://blog.openfoodfacts.org/wp-content/uploads/2025/07/2-150x150.jpg)


We're happy to announce that more than 1000 prices were added by contributors during the last two months. This document highlights some stats about the data.

In [164]:
import pandas as pd
df = pd.read_parquet("Summer_challenge_data.parquet")

### General statistics

In [165]:
print(f'{len(df.index)} prices, {df.product_code.nunique()} products, {pd.concat([df.owner_x, df.proof_owner]).nunique()} contributors')
print(f'{df.is_sunscreen.sum()} sunscreens prices, {len(df.index) - df.is_sunscreen.sum()} ice creams prices')
print(f'{df[df.is_sunscreen == 1].product_code.nunique()} sunscreens products, {df[df.is_sunscreen == 0].product_code.nunique()} ice creams products')
print(f'{df[df.created_t > 1751328000].product_code.nunique()} new products added to Open Food Facts and Open Beauty Facts')

1025 prices, 575 products, 34 contributors
282 sunscreens prices, 743 ice creams prices
119 sunscreens products, 456 ice creams products
108 new products added to Open Food Facts and Open Beauty Facts


### Top 10 prices contributors

In [166]:
df.groupby(['owner_x']).agg({'price': 'count'}).sort_values('price', ascending=False).head(10)

,price
owner_x,
un,166
jbieber,143
sebleouf,142
freso,123
poslovitch,105
ompopo,95
g123k,39
raphael0202,37
tacite,36


### Prices created using contributor photos

In [167]:
df.groupby(['proof_owner']).agg({'price': 'count'}).sort_values('price', ascending=False).head(10)

,price
proof_owner,
jbieber,167
sebleouf,144
freso,123
un,120
poslovitch,105
ompopo,96
bcd4e6,59
g123k,39
tacite,36


### Map of contributions

In [168]:
import folium
from folium.plugins import MarkerCluster

m = folium.Map(location=[df['osm_lat'].mean(), df['osm_lon'].mean()], zoom_start=2)
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.dropna(subset=['osm_lat', 'osm_lon']).iterrows():
    folium.Marker(
        location=[row['osm_lat'], row['osm_lon']],
        popup=f"price_id: {row['id']}, photo by: {row['proof_owner']}",
    ).add_to(marker_cluster)

m

### Tools used to contribute

In [169]:

def simple_source(source):
    if "Smoothie" in source:
        return "Smoothie"
    else:
        return source

df["simple_price_source"] = df.apply(lambda row: simple_source(row["source"]), axis=1)
df["simple_proof_source"] = df.apply(lambda row: simple_source(row["proof_source"]), axis=1)
df["proof_id"] = df.dump.apply(lambda row: json.loads(row)["proof_id"])

display(df.groupby("simple_price_source").agg({"id": "nunique"}).sort_values("id", ascending=False).reset_index().rename(columns={"simple_price_source": "Prices created using", "id": "Number of prices"}))

display(df.groupby("simple_proof_source").agg({"proof_id": "nunique"}).sort_values("proof_id", ascending=False).reset_index().rename(columns={"simple_proof_source": "Picture proof added using", "proof_id": "Number of pictures"}))

,Prices created using,Number of prices
0,Open Prices Web App - /experiments/price-valid...,474
1,Open Prices Web App - /prices/add/multiple,290
2,Smoothie,174
3,Open Prices Web App - /experiments/contributio...,43
4,Open Prices Web App - /experiments/receipt-ass...,28
5,Open Prices Web App - /experiments/proof-price...,15
6,Open Prices Web App - /prices/add/single,1


,Picture proof added using,Number of pictures
0,Open Prices Web App - /proofs/add/multiple,344
1,Smoothie,156
2,Open Prices Web App - /prices/add/multiple,71
3,Open Prices Web App - /experiments/receipt-ass...,30


### Sunscreens prices per litre

In [170]:
def convert_price(entry):
    match entry["currency"]:
        case "EUR":
            return entry["price"]
        case "SEK":
            return entry["price"] * 0.091
        case "NOK":
            return entry["price"] * 0.085
        case "CHF":
            return entry["price"] * 1.07
        case "RUB":
            return entry["price"] * 0.01
        case "GBP":
            return entry["price"] * 1.15
        case "PLN":
            return entry["price"] * 0.24
        case "USD":
            return entry["price"] * 0.85
        case _:
            return entry["price"]

df["price_in_euro"] = df.apply(lambda row: convert_price(row), axis=1)
df["euro_per_quantity"] = df.price_in_euro / df.product_quantity_x * 1000

In [171]:
avg_prices = df[(df.is_sunscreen == 1) & (df.product_quantity_x.notna())].groupby(["product_code"]).agg({"price_in_euro": "mean", "euro_per_quantity": "mean", "product_quantity_x": "first", "product_name_y": "first", "brands": "first"})
avg_prices["price_per_L"] = avg_prices["euro_per_quantity"]
avg_prices["readable_name"] = avg_prices.brands + ", " + avg_prices.product_name_y.apply(lambda x: x[0]["text"] if len(x) > 0 else "")
avg_prices.sort_values("price_per_L", ascending=False).reset_index()[["readable_name", "product_code", "price_per_L", "price_in_euro", "product_quantity_x"]]

,readable_name,product_code,price_per_L,price_in_euro,product_quantity_x
0,"ACO, Sun Lips SPF30",7319861023640,1797.250000,7.18900,4.0
1,"Nivea, SPF 30 sun protect",4005900983947,704.112500,2.81645,4.0
2,"Nivea Sun, Sun Lip Protect",4005900551269,611.975000,2.44790,4.0
3,"Acroelle, Crème solaire bébé spf 50+",3700343046082,367.000000,18.35000,50.0
4,"Acroelle, Face sunscreen",3700343046112,367.000000,18.35000,50.0
...,...,...,...,...,...
91,"Rema 1000, Solkrem, SPF30",7032069103432,37.825000,7.56500,200.0
92,"Nivea, Spray après soleil",4005900694003,35.800000,7.16000,200.0
93,"Derma, Sun Lotion Kids, SPF50",5709954039719,33.433333,5.01500,150.0
94,"Apoteket, Sollotion högt skydd SPF 30",7313272135268,31.622500,6.32450,200.0


### Average price per kg or L for each category of product
As expected, beauty products are more expensive than food products. 

Ice creams tend to be 40% more expensive than sorbets.

More surprisingly, ice creams cones tend to be cheaper than ice creams tubs per kg.

In [172]:
import numpy as np
df_exploded = df[df.euro_per_quantity != np.inf].explode('categories_tags')
average_prices = df_exploded.groupby('categories_tags')['euro_per_quantity'].agg(['mean', 'count']).reset_index().sort_values('mean', ascending=False)
average_prices[average_prices["count"] > 10].rename(columns={"categories_tags": "category", "mean": "Average € per kg or L", "count": "Number of prices"})

,category,Average € per kg or L,Number of prices
44,en:face,375.949737,19
45,en:facial-creams,227.848833,15
46,en:facial-sunscreens,227.848833,15
119,en:suncare,128.870876,233
77,en:in-sun-protections,128.870876,233
121,en:sunscreen,127.579796,239
19,en:chocolate-candies,26.506748,13
29,en:cocoa-and-its-products,26.506748,13
35,en:confectioneries,23.620038,25
123,en:sweet-snacks,22.202364,45
